# Evaluate Term Dispersion Scores on the GENIA Corpus Data and Reproduce Results Reported in the Mannuscript 

Description: Evaluate the following term dispersion score/keyword extraction methods on the Genia corpus data:
- Inverse Document Frequency (IDF)
- Inverse Collection Frequency (ICF)
- Chi-square
- Church and Gale (CG)
- Irvine and Callison-Burch (ICB)
- Derivation of Proportions (DoP)
- Residual ICF (RICF)
- KeyBERT
- KeyLLM

Calculate average P@k scores for each scoring function using the GENIA terms as ground truth. Also, evaluate scoring functions for their ability to filter out stopwords.

This version of the code includes singletons in the analysis.

## Preliminaries

In [1]:
# Imports
import sys
import os
import pickle
import json
import pandas as pd
sys.path.append('../../')
import wordstats
from sklearn.feature_extraction.text import CountVectorizer
import random
import numpy as np
import scipy
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from io import StringIO
from numpy import nan
from tqdm import tqdm
import rbo

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/pasheridan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Load the GENIA Corpus Data

In particular, we load the preprocessed GENIA corpus documents, and gold standard biological terms (i.e., lexical units) and their associated semantic classes (i.e., sems) and associated high-level class (i.e., amino_acid, nucleotide, multi_cell, cell, and other).

First, load the corpus docs, and the lexical units. Then hardcode the high-level semantic classes.

In [2]:
# Load the preprocessed GENIA corpus documents
genia_corpus_path = '../1-preprocessing/GENIAcorpus3.02-preprocessed.json'

with open(genia_corpus_path, "r") as j:
  genia_corpus = json.loads(j.read())

# Load gold standard terms 
genia_keywords_path = '../1-preprocessing/GENIAcorpus3.02-keywords.tsv'

with open(genia_keywords_path, "r") as c:
  genia_lexical_units_and_sems = pd.read_csv(c, sep='\t')

genia_lexical_units = genia_lexical_units_and_sems.lex.to_numpy()

# Hardcode the low-level semantic classes and their associated high-level abstract semantic classes
amino_acid_sems = ['G#amino_acid_monomer', 'G#peptide', 'G#protein_N/A',
              'G#protein_complex', 'G#protein_domain_or_region',
              'G#protein_family_or_group', 'G#protein_molecule',
              'G#protein_substructure', 'G#protein_subunit',
              'G#other_organic_compound', 'G#organic', 'G#inorganic', 'G#atom',
              'G#carbohydrate', 'G#lipid']
nucleotide_sems = ['G#nucleotide', 'G#polynucleotide', 'G#DNA_N/A',
        'G#DNA_domain_or_region', 'G#DNA_family_or_group', 'G#DNA_molecule',
        'G#DNA_substructure', 'G#RNA_N/A', 'G#RNA_domain_or_region',
        'G#RNA_family_or_group', 'G#RNA_molecule', 'G#RNA_substructure']
multi_cell_sems = ['G#virus', 'G#mono_cell', 'G#multi_cell', 'G#body_part', 'G#tissue']
cell_sems = ['G#cell_type', 'G#cell_component', 'G#cell_line', 'G#other_artificial_source']
other_sems = ['G#other_name']
high_level_semantic_class_names = ['amino_acid', 'nucleotide', 'multi_cell', 'cell', 'other']
high_level_semantic_class_lex_units = [genia_lexical_units, amino_acid_sems, nucleotide_sems, multi_cell_sems, cell_sems, other_sems]

Process high-level semantic classes.

In [3]:
# Collect lexical units belonging to a given high-level semantic class
def get_high_level_semantic_class_words(high_level_class_lst):
  words = []
  for k, v in lex_sem_dct.items():
    if v in high_level_class_lst:
      words.append(k)
  return words

# Create dictionary of lexical units and their associated semantic classes
sem = np.array(genia_lexical_units_and_sems['sem'])
lex = np.array(genia_lexical_units_and_sems['lex'])
lex_sem_dct = dict(zip(lex, sem))

# Create data frame of lexical units, semantic classes, and high-level semantic classes
lex_size = len(genia_lexical_units_and_sems) # Number of lexical units in the vocabulary
high_level_sems_lst = [] # Initialize list for recording high-level semantic classes

# For each term in the vocab, identify low-level semantic class with high-level one 
for index in range(lex_size):
    low_level_sem = genia_lexical_units_and_sems.iloc[index, 1]
    if low_level_sem in amino_acid_sems:
        high_level_sems_lst.append('amino_acid')
    elif low_level_sem in nucleotide_sems:
        high_level_sems_lst.append('nucleotide')
    elif low_level_sem in multi_cell_sems:
        high_level_sems_lst.append('multi_cell')
    elif low_level_sem in cell_sems:
        high_level_sems_lst.append('cell')
    else:
        high_level_sems_lst.append('other')

# Add high-level semantic classes to data frame
genia_lexical_units_and_sems['class'] = high_level_sems_lst

# Print to console:
display(genia_lexical_units_and_sems)

,lex,sem,class
0,IL-2_gene_expression_lex,G#other_name,other
1,IL-2_gene_lex,G#DNA_domain_or_region,nucleotide
2,NF-kappa_B_activation_lex,G#other_name,other
3,NF-kappa_B_lex,G#protein_molecule,amino_acid
4,CD28_lex,G#protein_molecule,amino_acid
...,...,...,...
31782,gp160-induced_AP-1_complex_lex,G#protein_complex,amino_acid
31783,protein_synthesis-independent_lex,G#other_name,other
31784,calcium_channel_blocker_lex,G#other_organic_compound,amino_acid
31785,anti-CD3-induced_interleukin-2_secretion_lex,G#other_name,other


## Prepare the GENIA Corpus Data for Analysis

Prepare the corpus vocabulary.

In [4]:
# Compile the GENIA corpus vocabulary
pre_vocab = []
for i in range(len(genia_corpus)):
  pre_vocab.append(genia_corpus[i].split())

vocab = []
for i in range(len(pre_vocab)):
  for j in range(len(pre_vocab[i])):
    vocab.append(pre_vocab[i][j])

vocab = list(set(vocab))
vocab.sort()

# Helper function to ensure that CountVectorizer doesn't ignore any terms
def analyzer_custom(doc):
  return doc.split()

# Convert GENIA documents into term-in-document matrix of token counts.
counter = CountVectorizer(lowercase=False, vocabulary=vocab, analyzer=analyzer_custom)
collection = counter.transform(genia_corpus)

## Evaluate Term Dispersion Scores for Selected Measures

Calculate bag-of-words model word statistics and related quantities.

In [5]:
# Calculate word statistics and related quantities
m = len(counter.get_feature_names_out()) # vocab size
d = collection.shape[0] # collection size
N_i = wordstats.get_Ni(collection)
N_j = wordstats.get_Nj(collection)
N = wordstats.get_N(N_j)
B_ij = wordstats.get_Bij(collection)
B_i = wordstats.get_Bi(B_ij)
B_j = wordstats.get_Bj(B_ij)
DF = wordstats.get_DF(B_i, d)
CF = wordstats.get_CF(N_i)
nij_by_nj = wordstats.get_nij_by_nj(collection, N_j)
thetas = np.array(range(1, max(N_i.A[0]) + 1))/N
opt_thetas = wordstats.get_opt_thetas(N, m, d, N_i, N_j, B_i, thetas)

Evaluate term dispersion scores.

In [6]:
# Calculate word dispersion scores according the various measures used in this study
IDF = wordstats.get_IDF(DF)
ICF = wordstats.get_ICF(CF)
Chisq = wordstats.get_Chisq(collection)
CG = wordstats.get_CG(N_i, B_i)
ICB = wordstats.get_ICB(nij_by_nj, B_i)
DoP = wordstats.get_DoP(collection, N_i, N_j, N)
RICF = wordstats.get_RICF(opt_thetas, N, ICF)

/Users/pasheridan/Desktop/github-repos/bursty-term-measure/genia/2-tables/../../wordstats.py:209: RuntimeWarning: divide by zero encountered in log
  return -np.log(chisq_values)


Arrange term dispersion scores into a data frame.

In [7]:
# Initialize term dispersion scores data frame (augmented with ni and bi values)
term_scores_aug_df = pd.DataFrame(data=
                    {'lex': counter.get_feature_names_out(),
                     'IDF': IDF.A[0],
                     'ICF': ICF.A[0],
                     'Chi-sq': Chisq,
                     'CG': CG.A[0],
                     'ICB': ICB.A[0],
                     'DoP': DoP.A[0],
                     'RICF': RICF.A[0],
                     'bi': B_i.A[0],
                     'ni': N_i.A[0]})

# Augment with low-level and high-level semantic classes
term_scores_aug_df = pd.merge(term_scores_aug_df, genia_lexical_units_and_sems, on='lex', how='left')

# Tidy up the data frame
new_order = ['lex', 'sem', 'class', 'bi', 'ni', 'IDF', 'ICF', 'Chi-sq', 'CG', 'ICB', 'DoP', 'RICF'] # Define column ordering
term_scores_aug_df = term_scores_aug_df.reindex(columns=new_order)
term_scores_aug_df = term_scores_aug_df.rename(columns={'lex': 'term'}) # Rename 'lex' column to 'term'
all_duplicates = term_scores_aug_df[term_scores_aug_df.duplicated(keep='first')] # There are a few duplicate rows for some unknown reason
print("Duplicate rows:")
display(all_duplicates)
term_scores_aug_df = term_scores_aug_df.drop_duplicates() # Drop any duplicate rows

# Print to console
print("Term dispersion scores:")
display(term_scores_aug_df)

Duplicate rows:


,term,sem,class,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
16654,basic_helix-loop-helix_protein_lex,G#protein_family_or_group,amino_acid,4,6,6.214608,11.009194,310.072539,1.50,233.5,-0.001611,0.404383
29778,minus_clone_lex,G#cell_line,cell,1,2,7.600902,12.107806,311.074036,2.00,466.0,-0.000643,0.692896
32037,octamer_motif_lex,G#DNA_domain_or_region,nucleotide,8,18,5.521461,9.910582,inf,2.25,449.5,-0.004192,0.808753


Term dispersion scores:


,term,sem,class,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
0,'aged'_lymphocyte_lex,G#cell_type,cell,1,1,7.600902,12.800954,0.701595,1.0,244.0,-0.000673,-0.000251
1,'converted'_TCEd_motif_lex,G#DNA_domain_or_region,nucleotide,1,1,7.600902,12.800954,0.701595,1.0,195.0,-0.000538,-0.000251
2,'latency_I'_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,-0.000251
3,'latency_II'_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,-0.000251
4,'master_regulator_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,63.0,-0.000174,-0.000251
...,...,...,...,...,...,...,...,...,...,...,...,...
40802,zymogen_plasma_factor_X_lex,G#protein_family_or_group,amino_acid,1,1,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,-0.000251
40803,zymogen_plasma_factors_VII_lex,G#protein_family_or_group,amino_acid,1,1,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,-0.000251
40804,zymography_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,191.0,-0.000527,-0.000251
40805,zymosan-treated_cell_lex,G#cell_type,cell,1,1,7.600902,12.800954,0.701595,1.0,225.0,-0.000621,-0.000251


In [8]:
# Integrate KeyBERT scores

# Load KeyBERT results 
keybert_keywords_path = 'keybert-scores.tsv'

with open(keybert_keywords_path, "r") as c:
  keybert_scores_df = pd.read_csv(c, sep='\t')

keybert_scores_df['sem'] = keybert_scores_df['sem'].fillna(str())

# Add KeyBERT scores to main results df
term_scores_aug_df = pd.merge(term_scores_aug_df, keybert_scores_df, on=['term', 'sem'], how='left')
term_scores_aug_df['KeyBERT'] = term_scores_aug_df['KeyBERT'].fillna(0)

# Check for duplicate terms
all_duplicates = keybert_scores_df[keybert_scores_df.duplicated(subset=['term', 'sem'], keep='last')]
print("Duplicate rows:")
display(all_duplicates)
keybert_scores_df = keybert_scores_df.drop_duplicates(subset=['term', 'sem'], keep='last')

# Reorder columns
term_scores_aug_df = term_scores_aug_df.reindex(columns=['term', 'sem', 'class', 'bi', 'ni', 'IDF', 'ICF', 'Chi-sq', 'CG', 'ICB', 'DoP', 'KeyBERT', 'RICF'])

# Check for duplicate terms
all_duplicates = term_scores_aug_df[term_scores_aug_df.duplicated(keep='first')]
print("Duplicate rows:")
display(all_duplicates)
term_scores_aug_df = term_scores_aug_df.drop_duplicates() # Drop any duplicate rows

display(term_scores_aug_df)

Duplicate rows:


,term,sem,KeyBERT
3132,basic_helix-loop-helix_protein_lex,G#protein_family_or_group,0.0010
16199,minus_clone_lex,G#cell_line,0.0005
18608,octamer_motif_lex,G#DNA_domain_or_region,0.0035


Duplicate rows:


,term,sem,class,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,RICF
16654,basic_helix-loop-helix_protein_lex,G#protein_family_or_group,amino_acid,4,6,6.214608,11.009194,310.072539,1.50,233.5,-0.001611,0.0010,0.404383
29778,minus_clone_lex,G#cell_line,cell,1,2,7.600902,12.107806,311.074036,2.00,466.0,-0.000643,0.0005,0.692896
32037,octamer_motif_lex,G#DNA_domain_or_region,nucleotide,8,18,5.521461,9.910582,inf,2.25,449.5,-0.004192,0.0035,0.808753


,term,sem,class,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,RICF
0,'aged'_lymphocyte_lex,G#cell_type,cell,1,1,7.600902,12.800954,0.701595,1.0,244.0,-0.000673,0.0000,-0.000251
1,'converted'_TCEd_motif_lex,G#DNA_domain_or_region,nucleotide,1,1,7.600902,12.800954,0.701595,1.0,195.0,-0.000538,0.0000,-0.000251
2,'latency_I'_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,0.0000,-0.000251
3,'latency_II'_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,0.0000,-0.000251
4,'master_regulator_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,63.0,-0.000174,0.0000,-0.000251
...,...,...,...,...,...,...,...,...,...,...,...,...,...
40802,zymogen_plasma_factor_X_lex,G#protein_family_or_group,amino_acid,1,1,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,0.0005,-0.000251
40803,zymogen_plasma_factors_VII_lex,G#protein_family_or_group,amino_acid,1,1,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,0.0005,-0.000251
40804,zymography_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,191.0,-0.000527,0.0000,-0.000251
40805,zymosan-treated_cell_lex,G#cell_type,cell,1,1,7.600902,12.800954,0.701595,1.0,225.0,-0.000621,0.0005,-0.000251


In [9]:
# Integrate KeyLLM scores

# Load KeyLLM results 
keyllm_keywords_path = 'keyllm-scores.tsv'

with open(keyllm_keywords_path, "r") as c:
  keyllm_scores_df = pd.read_csv(c, sep='\t')

keyllm_scores_df['sem'] = keyllm_scores_df['sem'].fillna(str())

# Check for duplicate terms
all_duplicates = keyllm_scores_df[keyllm_scores_df.duplicated(subset=['term', 'sem'], keep='last')]
print("Duplicate rows:")
display(all_duplicates)
keyllm_scores_df = keyllm_scores_df.drop_duplicates(subset=['term', 'sem'], keep='last')

# Print to console
print("KeyLLM scores:")
display(keyllm_scores_df)

# Add KeyLLM scores to main results df
term_scores_aug_df = pd.merge(term_scores_aug_df, keyllm_scores_df, on=['term', 'sem'], how='left')
term_scores_aug_df['KeyLLM'] = term_scores_aug_df['KeyLLM'].fillna(0)

# Reorder columns
term_scores_aug_df = term_scores_aug_df.reindex(columns=['term', 'sem', 'class', 'bi', 'ni', 'IDF', 'ICF', 'Chi-sq', 'CG', 'ICB', 'DoP', 'KeyBERT', 'KeyLLM', 'RICF'])

# Check for duplicate terms
all_duplicates = term_scores_aug_df[term_scores_aug_df.duplicated(keep='first')]
print("Duplicate rows:")
display(all_duplicates)
term_scores_aug_df = term_scores_aug_df.drop_duplicates() # Drop any duplicate rows

display(term_scores_aug_df)

Duplicate rows:


,term,sem,KeyLLM
136,20-epi_analogue_lex,G#other_organic_compound,0.0005
137,20-Epi_analogue_lex,G#other_organic_compound,0.0005
177,25-dihydroxyvitamin_d3,,0.0015
192,25_dihydroxyvitamin_d3,,0.0005
413,9-cis_RA_lex,G#other_organic_compound,0.0005
...,...,...,...
18825,pRb_lex,G#protein_molecule,0.0010
18826,pRB_lex,G#protein_family_or_group,0.0010
22321,v-abl_lex,G#DNA_domain_or_region,0.0005
22322,v-Abl_lex,G#protein_molecule,0.0005


KeyLLM scores:


,term,sem,KeyLLM
0,(3H)_dexamethasone_lex,G#lipid,0.0005
1,(Ca2+)i_lex,G#inorganic,0.0005
2,(Ca2+)i_requirement_for_lex,G#other_name,0.0005
3,-120_region_lex,G#DNA_domain_or_region,0.0005
4,-130_AP-1-like_site_lex,G#DNA_domain_or_region,0.0005
...,...,...,...
33986,CD4_negative_T_cell_line_lex,G#cell_line,0.0000
33987,gp_160-induced_nuclear_extract_lex,G#cell_component,0.0000
33988,gp160-induced_AP-1_complex_lex,G#protein_complex,0.0000
33989,protein_synthesis-independent_lex,G#other_name,0.0000


Duplicate rows:


,term,sem,class,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF


,term,sem,class,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
0,'aged'_lymphocyte_lex,G#cell_type,cell,1,1,7.600902,12.800954,0.701595,1.0,244.0,-0.000673,0.0000,0.0,-0.000251
1,'converted'_TCEd_motif_lex,G#DNA_domain_or_region,nucleotide,1,1,7.600902,12.800954,0.701595,1.0,195.0,-0.000538,0.0000,0.0,-0.000251
2,'latency_I'_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,0.0000,0.0,-0.000251
3,'latency_II'_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,0.0000,0.0,-0.000251
4,'master_regulator_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,63.0,-0.000174,0.0000,0.0,-0.000251
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40799,zymogen_plasma_factor_X_lex,G#protein_family_or_group,amino_acid,1,1,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,0.0005,0.0,-0.000251
40800,zymogen_plasma_factors_VII_lex,G#protein_family_or_group,amino_acid,1,1,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,0.0005,0.0,-0.000251
40801,zymography_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,191.0,-0.000527,0.0000,0.0,-0.000251
40802,zymosan-treated_cell_lex,G#cell_type,cell,1,1,7.600902,12.800954,0.701595,1.0,225.0,-0.000621,0.0005,0.0,-0.000251


In [10]:
# Write scores data frame to TSV
term_scores_aug_df.to_csv('term-dispersion-scores-n1.tsv', sep='\t', index=False)

## Compile GENIA Corpus Summary Statistics

This is the result of Table 3 from the manuscript.

In [11]:
# Desginated ordering for the high-level semantic classes
high_level_semantic_class_ord = ['amino_acid', 'nucleotide', 'multi_cell', 'cell', 'other']

# Count number of semantic subclasses in each high-level class
subclass_counts = [len(amino_acid_sems), len(nucleotide_sems), len(multi_cell_sems), len(cell_sems), len(other_sems)]

# Count number of distinct lexical units in each high-level semantic class
lex_unit_counts = term_scores_aug_df.dropna(subset=['class']).groupby('class')['term'].nunique().reindex(high_level_semantic_class_ord).to_list()

# Count number of annotations associated with each high-level semantic class
annotation_counts = term_scores_aug_df.dropna(subset=['class']).groupby('class')['ni'].sum().reindex(high_level_semantic_class_ord).to_list()

# Count number of singletons associated with each high-level semantic class
singleton_counts = term_scores_aug_df[term_scores_aug_df['ni'] == 1].dropna(subset=['class']).groupby('class')['ni'].sum().reindex(high_level_semantic_class_ord).to_list()

# Initialize GENIA summary statistics data frame
genia_summary_stats_df = pd.DataFrame({
    'Semantic class': high_level_semantic_class_ord,
    'Sub-class': subclass_counts,
    'Unique terms': lex_unit_counts,
    'Annotations': annotation_counts,
    'Singletons': singleton_counts
})

# Print GENIA summary statistics to console
display(genia_summary_stats_df)

,Semantic class,Sub-class,Unique terms,Annotations,Singletons
0,amino_acid,15,10155,42478,6571
1,nucleotide,12,5574,11619,4115
2,multi_cell,5,1444,5247,961
3,cell,4,4051,11626,2956
4,other,1,10560,19999,8071


## Terminology Extraction Task Experiment

Here we reproduce the result of Tables 5, 6, 7, A1, A2, and A3 from the manuscript.

In [13]:
# Create a minimal data frame of term dispersion scores
term_scores_df = term_scores_aug_df
term_scores_df = term_scores_df.reset_index(drop=True) # Reinitialize row indices
term_scores_df = term_scores_df.drop(columns=['sem', 'class', 'ni', 'bi'])

# Print to console
display(term_scores_df)

# Write scores data frame to TSV
term_scores_aug_df.to_csv('term-dispersion-scores-minimal-n1.tsv', sep='\t', index=False)

,term,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
0,'aged'_lymphocyte_lex,7.600902,12.800954,0.701595,1.0,244.0,-0.000673,0.0000,0.0,-0.000251
1,'converted'_TCEd_motif_lex,7.600902,12.800954,0.701595,1.0,195.0,-0.000538,0.0000,0.0,-0.000251
2,'latency_I'_lex,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,0.0000,0.0,-0.000251
3,'latency_II'_lex,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,0.0000,0.0,-0.000251
4,'master_regulator_lex,7.600902,12.800954,0.701595,1.0,63.0,-0.000174,0.0000,0.0,-0.000251
...,...,...,...,...,...,...,...,...,...,...
40799,zymogen_plasma_factor_X_lex,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,0.0005,0.0,-0.000251
40800,zymogen_plasma_factors_VII_lex,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,0.0005,0.0,-0.000251
40801,zymography_lex,7.600902,12.800954,0.701595,1.0,191.0,-0.000527,0.0000,0.0,-0.000251
40802,zymosan-treated_cell_lex,7.600902,12.800954,0.701595,1.0,225.0,-0.000621,0.0005,0.0,-0.000251


Define various functions used in the analysis.

In [14]:
# Grab the top k terms
def top_k(dct, k):
  keys = dct.keys()
  values = []
  for key in keys:
    values.append(dct[key][:k])
  keys_values_pair = zip(keys, values)
  return dict(keys_values_pair)

# Count up terms
def count_words(lst, imp_words):
  counter = 0
  for x in lst:
    if x in imp_words:
      counter += 1
  return counter

# Randomly resort term dispersion scores data frame
def resort(term_scores_df):
  sorted_terms = []
  bursty_measure_names = term_scores_df.columns.values.tolist()[1:]

  for measure in bursty_measure_names:
      # Copy the data frame and add a random column
      temp_df = term_scores_df.copy()
      temp_df['random'] = np.random.rand(len(temp_df))
        
      # Sort by the measure and the random column
      sorted_df = temp_df[['term', measure, 'random']].sort_values(by=[measure, 'random'], ascending=[False, True])
        
      # Append the sorted terms to the list
      sorted_terms.append(np.array(sorted_df['term']))
        
      # Drop the random column from the temporary data frame
      temp_df.drop(columns='random', inplace=True)
    
  sorted_terms = np.array(sorted_terms)
  measure_term_pair = zip(bursty_measure_names, sorted_terms)
  sorted_measures = dict(measure_term_pair)
    
  return sorted_measures
    
# Calculate Precision at k scores
def calc_pk(lex_units, sorted_measures, k_values):
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  pk_dct = dict(zip(measures, counts))  
  for measure in pk_dct.keys():
      for k in k_values:
          pk_dct[measure].append(count_words(top_k(sorted_measures, k)[measure], lex_units)/k)
  result = pd.DataFrame(pk_dct, index=k_values)
  return result

# Calculate Recall at k scores
def calc_rk(lex_units, sorted_measures, k_values):
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  rk_dct = dict(zip(measures, counts))  
  for measure in rk_dct.keys():
      for k in k_values:
          rk_dct[measure].append(count_words(top_k(sorted_measures, k)[measure], lex_units)/len(lex_units))
  result = pd.DataFrame(rk_dct, index=k_values)
  return result

# Calculate F1 at k scores
def calc_fk(pk, rk):
  result = 2 * (pk * rk) / (pk + rk)
  result = result.fillna(0) # Nan scores are redefined as 0
  return result

# Calculate Rank Biased Overlap scores
def calc_rbo(sorted_measures, k_values):
  RICF = sorted_measures["RICF"]
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  rbo_dct = dict(zip(measures, counts))  
  for measure in rbo_dct.keys():
      for k in k_values:
          S = top_k(sorted_measures, k)[measure]
          T = RICF[0:k]
          rbo_dct[measure].append(rbo.RankingSimilarity(S, T).rbo())
  result = pd.DataFrame(rbo_dct, index=k_values)
  return result

# Calculate Rank Biased Overlap scores for each semantic class
def calc_rbo2(sorted_measures, categories):
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  rbo_dct = dict(zip(measures, counts))

  for measure in measures:
      l1 = sorted_measures[measure].tolist()
      
      for category, lex_units in categories.items():
          S = sorted(set(l1) & set(lex_units), key = l1.index)
          l2 = sorted_measures["RICF"].tolist()
          RICF = sorted(set(l2) & set(lex_units), key = l2.index)          
          rbo_dct[measure].append(rbo.RankingSimilarity(S, RICF).rbo()) 

  result = pd.DataFrame(rbo_dct, index=categories.keys())
  return result
    
# Calculate mean P@k, R@k, and F1@k scores
def calc_score_means(nested_list):
    result = []
    num_outer = len(nested_list)
    num_inner = len(nested_list[0])

    for i in range(num_inner):
        means = {}
        for column in nested_list[0][i].columns:
            values = [nested_list[outer][i][column].values for outer in range(num_outer)]
            mean_values = np.mean(values, axis=0)
            means[column] = mean_values
        result.append(pd.DataFrame(means, index=nested_list[0][i].index))
    return result

# Calculate standard deviations of P@k, R@k, and F1@k scores
def calc_score_sds(nested_list):
    result = []
    num_outer = len(nested_list)
    num_inner = len(nested_list[0])

    for i in range(num_inner):
        std_devs = {}
        for column in nested_list[0][i].columns:
            values = [nested_list[outer][i][column].values for outer in range(num_outer)]
            std_values = np.std(values, axis=0, ddof=1)
            std_devs[column] = std_values
        result.append(pd.DataFrame(std_devs, index=nested_list[0][i].index))
    return result

# Calculate mean RBO scores
def calc_mean_rbo_scores(scores_list):
  R = len(scores_list) # Number of replicates
  H = len(scores_list[0]) # Number of dispersion metrics
  d_metrics = scores_list[0].columns # Dispersion metrics by name
  k_values = scores_list[0].index # Top k values
  result = {}
    
  for d_metric in d_metrics:
    scores = [scores_list[r][d_metric].values for r in range(R)]
    mean_scores = np.mean(scores, axis=0)
    result[d_metric] = mean_scores
        
  return pd.DataFrame(result, index=k_values)

Evaluate Precision at k, Recall at k, F1 at k, and RBO scores.

In [15]:
# Initialize a random seed to ensure results can be replicated
np.random.seed(641369)

# Set number of replicates
R = 100 # To test, set to 5

# These are the Precision @ k, Recall @ k and Rank Biased Overlap scores
all_pk_scores = []
all_rk_scores = []
all_fk_scores = []
all_rbo_scores = []
all_rbo_scores2 = []

# Used as inputs for calculating the various scores
k_values = np.array([10, 50, 100, 500, 1000, 5000])
amino_acid = get_high_level_semantic_class_words(amino_acid_sems)
nucleotide = get_high_level_semantic_class_words(nucleotide_sems)
multi_cell = get_high_level_semantic_class_words(multi_cell_sems)
cell = get_high_level_semantic_class_words(cell_sems)
other = get_high_level_semantic_class_words(other_sems)
categories = {
    'all': genia_lexical_units,
    'amino_acid': amino_acid,
    'nucleotide': nucleotide,
    'multi_cell': multi_cell,
    'cell': cell,
    'other': other}

# Calculate evaluation metrics
for r in tqdm(range(R)):
    print('r =', r)
    pk_scores = []
    rk_scores = []
    fk_scores = []
    sorted_measures = resort(term_scores_df)
    
    for category, lex_units in categories.items():
        pk = calc_pk(lex_units, sorted_measures, k_values)
        pk_scores.append(pk)
        rk = calc_rk(lex_units, sorted_measures, k_values)
        rk_scores.append(rk)
        fk = calc_fk(pk, rk)
        fk_scores.append(fk)
    
    all_pk_scores.append(pk_scores)
    all_rk_scores.append(rk_scores)
    all_fk_scores.append(fk_scores)
    rbo_scores = calc_rbo(sorted_measures, k_values)
    all_rbo_scores.append(rbo_scores)
    rbo_scores2 = calc_rbo2(sorted_measures, categories)
    all_rbo_scores2.append(rbo_scores2)

  0%|                                                                                                                           | 0/100 [00:00<?, ?it/s]

r = 0


  1%|█                                                                                                               | 1/100 [03:21<5:32:29, 201.51s/it]

r = 1


  2%|██▏                                                                                                             | 2/100 [06:45<5:31:19, 202.85s/it]

...

 98%|████████████████████████████████████████████████████████████████████████████████████████████████████████████▊  | 98/100 [5:39:10<06:35, 197.72s/it]

r = 98


 99%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████▉ | 99/100 [5:42:33<03:19, 199.28s/it]

r = 99


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [5:45:59<00:00, 207.59s/it]


Save evaluation metrics as Pkl files.

In [27]:
# Ensure the directory exists
os.makedirs('scores-dump-n1', exist_ok=True)

# Write P@k scores to Pkl
with open('scores-dump-n1/all_pk_scores.pkl', 'wb') as file:
    pickle.dump(all_pk_scores, file)

# Write R@k scores to Pkl
with open('scores-dump-n1/all_rk_scores.pkl', 'wb') as file:
    pickle.dump(all_rk_scores, file)

# Write F1@k scores to Pkl
with open('scores-dump-n1/all_fk_scores.pkl', 'wb') as file:
    pickle.dump(all_fk_scores, file)

# Write RBO scores to Pkl
with open('scores-dump-n1/all_rbo_scores.pkl', 'wb') as file:
    pickle.dump(all_rbo_scores, file)

# Write RBO scores as calculated for each semantic class to Pkl
with open('scores-dump-n1/all_rbo_scores2.pkl', 'wb') as file:
    pickle.dump(all_rbo_scores2, file)

In [26]:
# Calculate mean P@k scores and write to CSV
all_pk_scores_means = calc_score_means(all_pk_scores)
os.makedirs('table-5-n1', exist_ok=True)
pd.DataFrame(all_pk_scores_means[0]).to_csv('table-5-n1/all-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[1]).to_csv('table-5-n1/amino_acid-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[2]).to_csv('table-5-n1/nucleotide-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[3]).to_csv('table-5-n1/multicell-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[4]).to_csv('table-5-n1/cell-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[5]).to_csv('table-5-n1/other-pk-means.csv', index=False)

# Calculate standard deviations for P@k scores and write to CSV
all_pk_scores_sds = calc_score_sds(all_pk_scores)
os.makedirs('table-a1-n1', exist_ok=True)
pd.DataFrame(all_pk_scores_sds[0]).to_csv('table-a1-n1/all-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[1]).to_csv('table-a1-n1/amino_acid-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[2]).to_csv('table-a1-n1/nucleotide-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[3]).to_csv('table-a1-n1/multicell-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[4]).to_csv('table-a1-n1/cell-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[5]).to_csv('table-a1-n1/other-pk-sds.csv', index=False)

In [18]:
# Display mean P@k scores and console
print("Mean P@k scores:")
with pd.option_context('display.precision', 4):
    display(all_pk_scores_means[0].round(4))
    display(all_pk_scores_means[1].round(4))
    display(all_pk_scores_means[2].round(4))
    display(all_pk_scores_means[3].round(4))
    display(all_pk_scores_means[4].round(4))
    display(all_pk_scores_means[5].round(4))

Mean P@k scores:


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.8670,0.8430,0.9560,1.0000,1.0000,0.9000,1.0,1.0,1.0000
50,0.8612,0.8616,0.9544,0.9600,1.0000,0.7724,1.0,1.0,1.0000
100,0.8588,0.8572,0.9513,0.9800,0.9800,0.7800,1.0,1.0,1.0000
500,0.8605,0.8496,0.9529,0.9836,0.9740,0.8252,1.0,1.0,0.9915
1000,0.8605,0.8512,0.9528,0.9815,0.9637,0.8537,1.0,1.0,0.9853
5000,0.8604,0.8507,0.9151,0.9280,0.8972,0.8753,1.0,1.0,0.9315


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.2850,0.2440,0.5440,1.0000,0.8000,0.1770,0.6000,0.6000,1.0000
50,0.2690,0.2506,0.5302,0.7536,0.7000,0.2170,0.4060,0.5594,0.7914
100,0.2629,0.2522,0.5316,0.8200,0.7448,0.1317,0.4200,0.5387,0.8300
500,0.2622,0.2464,0.5375,0.6878,0.6220,0.2200,0.4671,0.4754,0.6918
1000,0.2652,0.2460,0.5394,0.6445,0.5901,0.2528,0.4521,0.4594,0.6428
5000,0.2650,0.2471,0.4291,0.4283,0.4109,0.2801,0.3652,0.3927,0.4294


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.1470,0.1660,0.1540,0.0000,0.1000,0.0000,0.1000,0.0000,0.0000
50,0.1570,0.1564,0.1582,0.1000,0.1200,0.1200,0.0800,0.0400,0.1000
100,0.1578,0.1545,0.1599,0.1000,0.1000,0.1435,0.0700,0.0900,0.1100
500,0.1558,0.1559,0.1539,0.1416,0.1460,0.1363,0.0888,0.1166,0.1428
1000,0.1560,0.1553,0.1528,0.1345,0.1395,0.1402,0.1116,0.1221,0.1367
5000,0.1567,0.1542,0.1529,0.1553,0.1483,0.1527,0.1410,0.1445,0.1557


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0410,0.0300,0.0390,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0376,0.0340,0.0448,0.0400,0.0400,0.0800,0.1000,0.0406,0.0400
100,0.0376,0.0342,0.0429,0.0200,0.0200,0.0900,0.0900,0.0500,0.0200
500,0.0365,0.0348,0.0406,0.0335,0.0320,0.0562,0.0545,0.0380,0.0338
1000,0.0360,0.0359,0.0402,0.0367,0.0420,0.0530,0.0565,0.0394,0.0358
5000,0.0365,0.0359,0.0437,0.0438,0.0413,0.0468,0.0491,0.0483,0.0441


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.1090,0.1150,0.0770,0.0000,0.0000,0.0740,0.1000,0.2000,0.0000
50,0.1016,0.1140,0.0790,0.0064,0.0000,0.1554,0.1540,0.2000,0.0086
100,0.1040,0.1098,0.0801,0.0100,0.0200,0.1200,0.1700,0.1500,0.0100
500,0.1100,0.1099,0.0823,0.0388,0.0600,0.0900,0.1552,0.1608,0.0399
1000,0.1094,0.1113,0.0820,0.0596,0.0690,0.0920,0.1390,0.1426,0.0599
5000,0.1104,0.1109,0.0954,0.1001,0.0987,0.0946,0.1239,0.1234,0.1007


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.2850,0.2880,0.1420,0.0000,0.1000,0.6490,0.2000,0.2000,0.0000
50,0.2960,0.3066,0.1422,0.0600,0.1400,0.2000,0.2600,0.1600,0.0600
100,0.2965,0.3065,0.1368,0.0300,0.0952,0.2948,0.2500,0.1713,0.0300
500,0.2960,0.3025,0.1385,0.0819,0.1140,0.3227,0.2344,0.2091,0.0831
1000,0.2938,0.3027,0.1385,0.1063,0.1231,0.3158,0.2409,0.2365,0.1101
5000,0.2918,0.3025,0.1941,0.2003,0.1981,0.3010,0.3209,0.2912,0.2015


In [19]:
# Displaye standard deviation of P@k scores to console
print("P@k scores standard deviations:")
with pd.option_context('display.precision', 4):
    display(all_pk_scores_sds[0].round(4))
    display(all_pk_scores_sds[1].round(4))
    display(all_pk_scores_sds[2].round(4))
    display(all_pk_scores_sds[3].round(4))
    display(all_pk_scores_sds[4].round(4))
    display(all_pk_scores_sds[5].round(4))

P@k scores standard deviations:


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.1035,0.1174,0.0641,0.0000,0.0000,0.0000,0.0,0.0,0.0000
50,0.0488,0.0451,0.0289,0.0000,0.0000,0.0098,0.0,0.0,0.0000
100,0.0339,0.0328,0.0192,0.0000,0.0000,0.0000,0.0,0.0,0.0000
500,0.0142,0.0146,0.0083,0.0008,0.0000,0.0010,0.0,0.0,0.0010
1000,0.0110,0.0096,0.0050,0.0017,0.0010,0.0004,0.0,0.0,0.0011
5000,0.0045,0.0042,0.0007,0.0008,0.0001,0.0001,0.0,0.0,0.0004


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.1373,0.1395,0.1591,0.0000,0.0000,0.0423,0.0000,0.0000,0.0000
50,0.0635,0.0621,0.0789,0.0094,0.0000,0.0072,0.0092,0.0100,0.0100
100,0.0438,0.0393,0.0529,0.0000,0.0050,0.0038,0.0000,0.0051,0.0000
500,0.0199,0.0177,0.0200,0.0037,0.0000,0.0000,0.0020,0.0045,0.0041
1000,0.0146,0.0131,0.0125,0.0055,0.0007,0.0006,0.0042,0.0037,0.0039
5000,0.0057,0.0057,0.0009,0.0011,0.0002,0.0002,0.0024,0.0027,0.0005


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.1123,0.1233,0.1077,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0518,0.0538,0.0551,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
100,0.0379,0.0348,0.0342,0.0000,0.0000,0.0048,0.0000,0.0000,0.0000
500,0.0146,0.0151,0.0131,0.0024,0.0000,0.0007,0.0011,0.0029,0.0028
1000,0.0105,0.0115,0.0093,0.0036,0.0005,0.0008,0.0024,0.0028,0.0027
5000,0.0041,0.0045,0.0008,0.0008,0.0001,0.0002,0.0020,0.0021,0.0004


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0637,0.0522,0.0567,0.0000,0.0000,0.0000,0.000,0.0000,0.0000
50,0.0270,0.0230,0.0306,0.0000,0.0000,0.0000,0.000,0.0100,0.0000
100,0.0190,0.0180,0.0185,0.0000,0.0000,0.0000,0.000,0.0000,0.0000
500,0.0075,0.0079,0.0085,0.0015,0.0000,0.0007,0.001,0.0014,0.0017
1000,0.0053,0.0054,0.0052,0.0017,0.0007,0.0000,0.002,0.0016,0.0014
5000,0.0026,0.0020,0.0005,0.0005,0.0001,0.0000,0.001,0.0012,0.0003


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0965,0.1067,0.0750,0.0000,0.0000,0.0441,0.0000,0.0000,0.0000
50,0.0418,0.0460,0.0381,0.0094,0.0000,0.0085,0.0092,0.0000,0.0100
100,0.0290,0.0309,0.0284,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
500,0.0147,0.0129,0.0114,0.0021,0.0000,0.0000,0.0015,0.0024,0.0023
1000,0.0101,0.0091,0.0061,0.0024,0.0008,0.0000,0.0027,0.0024,0.0018
5000,0.0034,0.0035,0.0006,0.0007,0.0001,0.0001,0.0020,0.0020,0.0004


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.1373,0.1506,0.0934,0.0000,0.0000,0.0559,0.0000,0.0000,0.0000
50,0.0676,0.0669,0.0485,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
100,0.0468,0.0510,0.0339,0.0000,0.0050,0.0050,0.0000,0.0051,0.0000
500,0.0185,0.0206,0.0142,0.0024,0.0000,0.0010,0.0020,0.0042,0.0027
1000,0.0150,0.0131,0.0088,0.0035,0.0007,0.0006,0.0035,0.0035,0.0027
5000,0.0058,0.0059,0.0009,0.0010,0.0003,0.0002,0.0026,0.0025,0.0005


In [25]:
# Calculate mean R@k scores and write to CSV
all_rk_scores_means = calc_score_means(all_rk_scores)
os.makedirs('table-6-n1', exist_ok=True)
pd.DataFrame(all_rk_scores_means[0]).to_csv('table-6-n1/all-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[1]).to_csv('table-6-n1/amino_acid-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[2]).to_csv('table-6-n1/nucleotide-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[3]).to_csv('table-6-n1/multicell-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[4]).to_csv('table-6-n1/cell-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[5]).to_csv('table-6-n1/other-rk-means.csv', index=False)

# Calculate standard deviations for R@k scores and write to CSV
all_rk_scores_sds = calc_score_sds(all_rk_scores)
os.makedirs('table-a2-n1', exist_ok=True)
pd.DataFrame(all_rk_scores_sds[0]).to_csv('table-a2-n1/all-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[1]).to_csv('table-a2-n1/amino_acid-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[2]).to_csv('table-a2-n1/nucleotide-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[3]).to_csv('table-a2-n1/multicell-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[4]).to_csv('table-a2-n1/cell-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[5]).to_csv('table-a2-n1/other-rk-sds.csv', index=False)

In [21]:
# Display mean R@k scores console
print("Mean R@k scores:")
with pd.option_context('display.precision', 4):
    display(all_rk_scores_means[0].round(4))
    display(all_rk_scores_means[1].round(4))
    display(all_rk_scores_means[2].round(4))
    display(all_rk_scores_means[3].round(4))
    display(all_rk_scores_means[4].round(4))
    display(all_rk_scores_means[5].round(4))

Mean R@k scores:


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0003,0.0003,0.0003,0.0003,0.0003,0.0003,0.0003,0.0003,0.0003
50,0.0014,0.0014,0.0015,0.0015,0.0016,0.0012,0.0016,0.0016,0.0016
100,0.0027,0.0027,0.0030,0.0031,0.0031,0.0025,0.0031,0.0031,0.0031
500,0.0135,0.0134,0.0150,0.0155,0.0153,0.0130,0.0157,0.0157,0.0156
1000,0.0271,0.0268,0.0300,0.0309,0.0303,0.0269,0.0315,0.0315,0.0310
5000,0.1353,0.1338,0.1439,0.1460,0.1411,0.1377,0.1573,0.1573,0.1465


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0003,0.0002,0.0005,0.0010,0.0008,0.0002,0.0006,0.0006,0.0010
50,0.0013,0.0012,0.0026,0.0037,0.0034,0.0011,0.0020,0.0028,0.0039
100,0.0026,0.0025,0.0052,0.0081,0.0073,0.0013,0.0041,0.0053,0.0082
500,0.0129,0.0121,0.0265,0.0339,0.0306,0.0108,0.0230,0.0234,0.0341
1000,0.0261,0.0242,0.0531,0.0635,0.0581,0.0249,0.0445,0.0452,0.0633
5000,0.1305,0.1217,0.2113,0.2109,0.2023,0.1379,0.1798,0.1933,0.2114


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0003,0.0003,0.0003,0.0000,0.0002,0.0000,0.0002,0.0000,0.0000
50,0.0014,0.0014,0.0014,0.0009,0.0011,0.0011,0.0007,0.0004,0.0009
100,0.0028,0.0028,0.0029,0.0018,0.0018,0.0026,0.0013,0.0016,0.0020
500,0.0140,0.0140,0.0138,0.0127,0.0131,0.0122,0.0080,0.0105,0.0128
1000,0.0280,0.0279,0.0274,0.0241,0.0250,0.0251,0.0200,0.0219,0.0245
5000,0.1406,0.1383,0.1371,0.1393,0.1330,0.1370,0.1264,0.1296,0.1397


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0003,0.0002,0.0003,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0013,0.0012,0.0016,0.0014,0.0014,0.0028,0.0035,0.0014,0.0014
100,0.0026,0.0024,0.0030,0.0014,0.0014,0.0062,0.0062,0.0035,0.0014
500,0.0126,0.0121,0.0141,0.0116,0.0111,0.0195,0.0189,0.0132,0.0117
1000,0.0250,0.0249,0.0278,0.0254,0.0291,0.0367,0.0391,0.0273,0.0248
5000,0.1265,0.1244,0.1514,0.1517,0.1429,0.1620,0.1699,0.1673,0.1528


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0003,0.0003,0.0002,0.0000,0.0000,0.0002,0.0002,0.0005,0.0000
50,0.0013,0.0014,0.0010,0.0001,0.0000,0.0019,0.0019,0.0025,0.0001
100,0.0026,0.0027,0.0020,0.0002,0.0005,0.0030,0.0042,0.0037,0.0002
500,0.0136,0.0136,0.0102,0.0048,0.0074,0.0111,0.0192,0.0199,0.0049
1000,0.0270,0.0275,0.0202,0.0147,0.0170,0.0227,0.0343,0.0352,0.0148
5000,0.1362,0.1369,0.1177,0.1236,0.1219,0.1168,0.1529,0.1522,0.1243


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0003,0.0003,0.0001,0.0000,0.0001,0.0006,0.0002,0.0002,0.0000
50,0.0014,0.0015,0.0007,0.0003,0.0007,0.0009,0.0012,0.0008,0.0003
100,0.0028,0.0029,0.0013,0.0003,0.0009,0.0028,0.0024,0.0016,0.0003
500,0.0140,0.0143,0.0066,0.0039,0.0054,0.0153,0.0111,0.0099,0.0039
1000,0.0278,0.0287,0.0131,0.0101,0.0117,0.0299,0.0228,0.0224,0.0104
5000,0.1381,0.1433,0.0919,0.0949,0.0938,0.1425,0.1520,0.1379,0.0954


In [22]:
# Display standard deviation of R@k scores to console
print("R@k scores standard deviations:")
with pd.option_context('display.precision', 4):
    display(all_rk_scores_sds[0].round(4))
    display(all_rk_scores_sds[1].round(4))
    display(all_rk_scores_sds[2].round(4))
    display(all_rk_scores_sds[3].round(4))
    display(all_rk_scores_sds[4].round(4))
    display(all_rk_scores_sds[5].round(4))

R@k scores standard deviations:


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0000,0.0000,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000
50,0.0001,0.0001,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000
100,0.0001,0.0001,0.0001,0.0000,0.0,0.0,0.0,0.0,0.0000
500,0.0002,0.0002,0.0001,0.0000,0.0,0.0,0.0,0.0,0.0000
1000,0.0003,0.0003,0.0002,0.0001,0.0,0.0,0.0,0.0,0.0000
5000,0.0007,0.0007,0.0001,0.0001,0.0,0.0,0.0,0.0,0.0001


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0001,0.0001,0.0002,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0003,0.0003,0.0004,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
100,0.0004,0.0004,0.0005,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
500,0.0010,0.0009,0.0010,0.0002,0.0000,0.0000,0.0001,0.0002,0.0002
1000,0.0014,0.0013,0.0012,0.0005,0.0001,0.0001,0.0004,0.0004,0.0004
5000,0.0028,0.0028,0.0004,0.0005,0.0001,0.0001,0.0012,0.0013,0.0002


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0002,0.0002,0.0002,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0005,0.0005,0.0005,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
100,0.0007,0.0006,0.0006,0.0000,0.0000,0.0001,0.0000,0.0000,0.0000
500,0.0013,0.0014,0.0012,0.0002,0.0000,0.0001,0.0001,0.0003,0.0003
1000,0.0019,0.0021,0.0017,0.0006,0.0001,0.0001,0.0004,0.0005,0.0005
5000,0.0037,0.0040,0.0007,0.0007,0.0001,0.0002,0.0018,0.0019,0.0003


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0004,0.0004,0.0004,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0009,0.0008,0.0011,0.0000,0.0000,0.0000,0.0000,0.0003,0.0000
100,0.0013,0.0012,0.0013,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
500,0.0026,0.0027,0.0030,0.0005,0.0000,0.0002,0.0003,0.0005,0.0006
1000,0.0037,0.0038,0.0036,0.0012,0.0005,0.0000,0.0014,0.0011,0.0010
5000,0.0089,0.0070,0.0017,0.0018,0.0003,0.0000,0.0035,0.0042,0.0009


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0002,0.0003,0.0002,0.0000,0.0000,0.0001,0.0000,0.0000,0.0000
50,0.0005,0.0006,0.0005,0.0001,0.0000,0.0001,0.0001,0.0000,0.0001
100,0.0007,0.0008,0.0007,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
500,0.0018,0.0016,0.0014,0.0003,0.0000,0.0000,0.0002,0.0003,0.0003
1000,0.0025,0.0023,0.0015,0.0006,0.0002,0.0000,0.0007,0.0006,0.0004
5000,0.0042,0.0043,0.0008,0.0009,0.0002,0.0001,0.0025,0.0024,0.0004


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0001,0.0001,0.0001,0.0000,0.0000,0.0001,0.0000,0.0000,0.0000
50,0.0003,0.0003,0.0002,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
100,0.0004,0.0005,0.0003,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
500,0.0009,0.0010,0.0007,0.0001,0.0000,0.0000,0.0001,0.0002,0.0001
1000,0.0014,0.0012,0.0008,0.0003,0.0001,0.0001,0.0003,0.0003,0.0003
5000,0.0028,0.0028,0.0004,0.0005,0.0001,0.0001,0.0013,0.0012,0.0002


In [24]:
# Calculate mean F1@k scores and write to CSV
all_fk_scores_means = calc_score_means(all_fk_scores)
os.makedirs('table-7-n1', exist_ok=True)
pd.DataFrame(all_fk_scores_means[0]).to_csv('table-7-n1/all-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[1]).to_csv('table-7-n1/amino_acid-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[2]).to_csv('table-7-n1/nucleotide-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[3]).to_csv('table-7-n1/multicell-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[4]).to_csv('table-7-n1/cell-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[5]).to_csv('table-7-n1/other-fk-means.csv', index=False)

# Calculate standard deviations for F1@k scores and write to CSV
all_fk_scores_sds = calc_score_sds(all_fk_scores)
os.makedirs('table-a3-n1', exist_ok=True)
pd.DataFrame(all_fk_scores_sds[0]).to_csv('table-a3-n1/all-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[1]).to_csv('table-a3-n1/amino_acid-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[2]).to_csv('table-a3-n1/nucleotide-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[3]).to_csv('table-a3-n1/multicell-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[4]).to_csv('table-a3-n1/cell-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[5]).to_csv('table-a3-n1/other-fk-sds.csv', index=False)

In [28]:
# Display mean F1@k scores to console
print("Mean F1@k scores:")
with pd.option_context('display.precision', 4):
    display(all_fk_scores_means[0].round(4))
    display(all_fk_scores_means[1].round(4))
    display(all_fk_scores_means[2].round(4))
    display(all_fk_scores_means[3].round(4))
    display(all_fk_scores_means[4].round(4))
    display(all_fk_scores_means[5].round(4))

Mean F1@k scores:


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0005,0.0005,0.0006,0.0006,0.0006,0.0006,0.0006,0.0006,0.0006
50,0.0027,0.0027,0.0030,0.0030,0.0031,0.0024,0.0031,0.0031,0.0031
100,0.0054,0.0054,0.0060,0.0061,0.0061,0.0049,0.0063,0.0063,0.0063
500,0.0267,0.0263,0.0295,0.0305,0.0302,0.0256,0.0310,0.0310,0.0307
1000,0.0525,0.0519,0.0581,0.0599,0.0588,0.0521,0.0610,0.0610,0.0601
5000,0.2339,0.2312,0.2488,0.2523,0.2439,0.2379,0.2718,0.2718,0.2532


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0006,0.0005,0.0011,0.0020,0.0016,0.0003,0.0012,0.0012,0.0020
50,0.0026,0.0025,0.0052,0.0074,0.0069,0.0021,0.0040,0.0055,0.0078
100,0.0051,0.0049,0.0104,0.0160,0.0145,0.0026,0.0082,0.0105,0.0162
500,0.0246,0.0231,0.0504,0.0645,0.0584,0.0206,0.0438,0.0446,0.0649
1000,0.0476,0.0441,0.0967,0.1155,0.1058,0.0453,0.0810,0.0824,0.1152
5000,0.1748,0.1631,0.2831,0.2826,0.2711,0.1848,0.2410,0.2591,0.2833


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0005,0.0006,0.0006,0.0000,0.0004,0.0000,0.0004,0.0000,0.0000
50,0.0028,0.0028,0.0028,0.0018,0.0021,0.0021,0.0014,0.0007,0.0018
100,0.0056,0.0054,0.0056,0.0035,0.0035,0.0051,0.0025,0.0032,0.0039
500,0.0257,0.0257,0.0253,0.0233,0.0240,0.0224,0.0146,0.0192,0.0235
1000,0.0475,0.0472,0.0465,0.0409,0.0425,0.0426,0.0339,0.0371,0.0416
5000,0.1482,0.1458,0.1446,0.1469,0.1402,0.1444,0.1333,0.1367,0.1473


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0006,0.0004,0.0005,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0025,0.0023,0.0030,0.0027,0.0027,0.0054,0.0067,0.0027,0.0027
100,0.0049,0.0044,0.0056,0.0026,0.0026,0.0117,0.0117,0.0065,0.0026
500,0.0188,0.0179,0.0209,0.0172,0.0165,0.0289,0.0280,0.0196,0.0174
1000,0.0295,0.0294,0.0329,0.0300,0.0344,0.0434,0.0462,0.0323,0.0293
5000,0.0567,0.0558,0.0679,0.0680,0.0640,0.0726,0.0761,0.0750,0.0685


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0005,0.0006,0.0004,0.0000,0.0000,0.0004,0.0005,0.0010,0.0000
50,0.0025,0.0028,0.0019,0.0002,0.0000,0.0038,0.0038,0.0049,0.0002
100,0.0050,0.0053,0.0039,0.0005,0.0010,0.0058,0.0082,0.0072,0.0005
500,0.0242,0.0241,0.0181,0.0085,0.0132,0.0198,0.0341,0.0353,0.0088
1000,0.0433,0.0441,0.0325,0.0236,0.0273,0.0364,0.0550,0.0564,0.0237
5000,0.1220,0.1225,0.1054,0.1106,0.1091,0.1046,0.1369,0.1363,0.1112


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0005,0.0005,0.0003,0.0000,0.0002,0.0012,0.0004,0.0004,0.0000
50,0.0028,0.0029,0.0013,0.0006,0.0013,0.0019,0.0025,0.0015,0.0006
100,0.0056,0.0058,0.0026,0.0006,0.0018,0.0055,0.0047,0.0032,0.0006
500,0.0268,0.0274,0.0125,0.0074,0.0103,0.0292,0.0212,0.0189,0.0075
1000,0.0508,0.0524,0.0240,0.0184,0.0213,0.0546,0.0417,0.0409,0.0190
5000,0.1875,0.1944,0.1247,0.1288,0.1273,0.1935,0.2062,0.1871,0.1295


In [29]:
# Display standard deviation of F1@k scores to console
print("F1@k scores standard deviations:")
with pd.option_context('display.precision', 4):
    display(all_fk_scores_sds[0].round(4))
    display(all_fk_scores_sds[1].round(4))
    display(all_fk_scores_sds[2].round(4))
    display(all_fk_scores_sds[3].round(4))
    display(all_fk_scores_sds[4].round(4))
    display(all_fk_scores_sds[5].round(4))

F1@k scores standard deviations:


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0001,0.0001,0.0000,0.0000,0.0000,0.0,0.0,0.0,0.0000
50,0.0002,0.0001,0.0001,0.0000,0.0000,0.0,0.0,0.0,0.0000
100,0.0002,0.0002,0.0001,0.0000,0.0000,0.0,0.0,0.0,0.0000
500,0.0004,0.0005,0.0003,0.0000,0.0000,0.0,0.0,0.0,0.0000
1000,0.0007,0.0006,0.0003,0.0001,0.0001,0.0,0.0,0.0,0.0001
5000,0.0012,0.0011,0.0002,0.0002,0.0000,0.0,0.0,0.0,0.0001


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0003,0.0003,0.0003,0.0000,0.0000,0.0001,0.0000,0.0000,0.0000
50,0.0006,0.0006,0.0008,0.0001,0.0000,0.0001,0.0001,0.0001,0.0001
100,0.0009,0.0008,0.0010,0.0000,0.0001,0.0001,0.0000,0.0001,0.0000
500,0.0019,0.0017,0.0019,0.0003,0.0000,0.0000,0.0002,0.0004,0.0004
1000,0.0026,0.0023,0.0022,0.0010,0.0001,0.0001,0.0008,0.0007,0.0007
5000,0.0038,0.0037,0.0006,0.0007,0.0002,0.0001,0.0016,0.0018,0.0003


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0004,0.0004,0.0004,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0009,0.0010,0.0010,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
100,0.0013,0.0012,0.0012,0.0000,0.0000,0.0002,0.0000,0.0000,0.0000
500,0.0024,0.0025,0.0022,0.0004,0.0000,0.0001,0.0002,0.0005,0.0005
1000,0.0032,0.0035,0.0028,0.0011,0.0002,0.0002,0.0007,0.0009,0.0008
5000,0.0038,0.0042,0.0008,0.0008,0.0001,0.0002,0.0019,0.0020,0.0003


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0009,0.0007,0.0008,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0018,0.0015,0.0021,0.0000,0.0000,0.0000,0.0000,0.0007,0.0000
100,0.0025,0.0023,0.0024,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
500,0.0039,0.0040,0.0044,0.0008,0.0000,0.0003,0.0005,0.0007,0.0009
1000,0.0043,0.0045,0.0042,0.0014,0.0005,0.0000,0.0016,0.0013,0.0012
5000,0.0040,0.0031,0.0007,0.0008,0.0001,0.0000,0.0016,0.0019,0.0004


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0005,0.0005,0.0004,0.0000,0.0000,0.0002,0.0000,0.0000,0.0000
50,0.0010,0.0011,0.0009,0.0002,0.0000,0.0002,0.0002,0.0000,0.0002
100,0.0014,0.0015,0.0014,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
500,0.0032,0.0028,0.0025,0.0005,0.0000,0.0000,0.0003,0.0005,0.0005
1000,0.0040,0.0036,0.0024,0.0009,0.0003,0.0000,0.0011,0.0009,0.0007
5000,0.0037,0.0039,0.0007,0.0008,0.0001,0.0001,0.0022,0.0022,0.0004


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0003,0.0003,0.0002,0.0000,0.0000,0.0001,0.0000,0.0000,0.0000
50,0.0006,0.0006,0.0005,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
100,0.0009,0.0010,0.0006,0.0000,0.0001,0.0001,0.0000,0.0001,0.0000
500,0.0017,0.0019,0.0013,0.0002,0.0000,0.0001,0.0002,0.0004,0.0002
1000,0.0026,0.0023,0.0015,0.0006,0.0001,0.0001,0.0006,0.0006,0.0005
5000,0.0037,0.0038,0.0006,0.0006,0.0002,0.0001,0.0017,0.0016,0.0003


In [30]:
# Calculate mean RBO scores
mean_rbo_scores = calc_mean_rbo_scores(all_rbo_scores)

# Write to CSV
os.makedirs('table-8-n1', exist_ok=True)
pd.DataFrame(mean_rbo_scores).to_csv('table-8-n1/rbo-means.csv', index=False)

# Display RBO scores to console
with pd.option_context('display.precision', 4):
    display(mean_rbo_scores.round(4))

,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0000,0.0,0.0002,0.9154,0.6133,0.0000,0.0000,0.0000,1.0
50,0.0004,0.0,0.0082,0.8990,0.6866,0.0000,0.0000,0.0000,1.0
100,0.0010,0.0,0.0189,0.9086,0.7023,0.0000,0.0000,0.0000,1.0
500,0.0071,0.0,0.0997,0.9314,0.7287,0.0005,0.0019,0.0011,1.0
1000,0.0127,0.0,0.1979,0.9358,0.7654,0.0030,0.0148,0.0123,1.0
5000,0.0606,0.0,0.6529,0.9067,0.7954,0.0430,0.1174,0.1179,1.0


In [31]:
# Calculate mean RBO scores by semantic class
mean_rbo_scores2 = calc_mean_rbo_scores(all_rbo_scores2)

# Write to CSV
os.makedirs('table-9-n1', exist_ok=True)
pd.DataFrame(mean_rbo_scores2).to_csv('table-9-n1/rbo-means-by-semantic-class.csv', index=False)

# Display RBO scores to console
with pd.option_context('display.precision', 4):
    display(mean_rbo_scores2.round(4))

,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
all,0.4991,0.4269,0.7824,0.7934,0.7487,0.4904,0.5330,0.5395,1.0
amino_acid,0.4832,0.3866,0.7978,0.8416,0.7961,0.4728,0.5385,0.5490,1.0
nucleotide,0.5035,0.4322,0.7707,0.7793,0.7406,0.4914,0.5262,0.5322,1.0
multi_cell,0.4989,0.4164,0.8049,0.8111,0.7541,0.4881,0.5385,0.5305,1.0
cell,0.5010,0.4360,0.7789,0.7778,0.7300,0.4926,0.5287,0.5296,1.0
other,0.5142,0.4636,0.7560,0.7402,0.6934,0.5077,0.5260,0.5253,1.0


## Top 10 Ranked Terms Example

Here we reproduce the result of Table 10 from the manuscript.

In [32]:
# Initialize a random seed to ensure results can be replicated
np.random.seed(641369)

# Retrieve top 10 ranked terms
top = 10
ranked_terms_df = resort(term_scores_df)
top_10_ranked_terms_df = pd.DataFrame(top_k(ranked_terms_df, top))

# Print to console
display(top_10_ranked_terms_df)

# Write to CSV
os.makedirs('table-10-n1', exist_ok=True)
top_10_ranked_terms_df.to_csv('table-10-n1/top-10-terms.csv', index=False)

,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
0,recessive,CLE0_element_lex,P_sequence_lex,Bcl-6_lex,Bcl-6_lex,guardian,transcription_factor_lex,transcription_factor_lex,Bcl-6_lex
1,helix,RAP30_lex,K_protein_lex,SMX_lex,TCRzeta_lex,distinct_function_lex,activation_lex,T_cell_lex,SMX_lex
2,P3A2_lex,12-O-tetradecanoylphorbol-13-acetate_(TPA)_res...,UAS2_lex,v-erbA_lex,ML-9_lex,interleukin-15_lex,cytokine_lex,NF-kappa_b_lex,v-erbA_lex
3,p0005,AIDS_IBLP_tumor_lex,Bik_lex,ML-9_lex,AITL_lex,thymus-derived_T-cell_homeostasis_lex,transcription_lex,NF-kappa_B_lex,SHP1_lex
4,obese_patient_lex,beta-like_globin_cluster_lex,sesquiterpene_lactone_lex,SHP1_lex,SHP1_lex,extrathymic_development_lex,T_cell_lex,cytokine_lex,ML-9_lex
5,human_activated_monocyte_lex,stromal-derived_cytokine_interleukin-7_lex,TS_lex,beta-casein_lex,beta-casein_lex,B-lymphocyte_differentiation_control_lex,gene_lex,NF-kappaB_lex,beta-casein_lex
6,putative_chicken_Shc_homologue_lex,inductive_stimulation_lex,MTBE_lex,EBNA-2_lex,A-myb_lex,B-lymphocyte_differentiation_lex,NF-kappa_b_lex,AP-1_lex,p95vav_lex
7,DNA_ploidy_lex,Myc_lex,Tax_lex,DM_lex,I_kappaB_lex,plasma_cell_pathway_lex,NF-kappa_B_lex,transcription_lex,I_kappaB_lex
8,mature_organ_lex,N-terminal_c-Jun_kinase_lex,variants,TCRzeta_lex,SMX_lex,B-cell_commitment_lex,Il-2_lex,gene_expression_lex,TCRzeta_lex
9,small_GTP-binding_protein_Rho_lex,PKR_expression_lex,glial_cell_lex,I_kappaB_lex,Rap1_protein_lex,memory_B_cells_lex,IL-2_lex,T_lymphocyte_lex,DM_lex


## Stopwords Exploratory Analysis

Here we reproduce the result of Table 11 from the manuscript.

In [33]:
def getrank(sorted_measures):
    unique_terms = set()
    for terms in sorted_measures.values():
        unique_terms.update(terms)
    unique_terms = sorted(unique_terms)
    
    # Create a data frame to hold the rankings
    ranking_df = pd.DataFrame(index=unique_terms, columns=sorted_measures.keys())
    
    # Fill the data frame with rankings
    for measure, terms in sorted_measures.items():
        for rank, term in enumerate(terms):
            ranking_df.at[term, measure] = rank + 1  # Rank starts from 1
    
    # Replace NaN with a large number to indicate unranked terms
    ranking_df = ranking_df.fillna(len(unique_terms) + 1)
    #csv_file_path = 'ranking_table.csv'
    #ranking_df.to_csv(csv_file_path)
    return ranking_df

# Function to filter stopwords from the ranking data frame
def filter_stopwords(ranking_df):
    stopwords_list = set(stopwords.words('english'))
    
    # Filter the data frame to include only stopwords
    stopwords_rank = ranking_df[ranking_df.index.isin(stopwords_list)]
    
    # Save the stopwords ranking data frame to a CSV file
    #csv_file_path = 'stopwords_ranking_table.csv'
    #stopwords_rank.to_csv(csv_file_path)
    
    return stopwords_rank

In [34]:
# Initialize a random seed to ensure results can be replicated
np.random.seed(641369)

# Generate term dispersion ranks for R different versions of the data
all_quantiles_df = []
for r in tqdm(range(R)):
    sorted_measures = resort(term_scores_df)
    rank = getrank(sorted_measures)
    stopwords_ranks_df = filter_stopwords(rank)
    bursty_measure_names = stopwords_ranks_df.head(0)
    quantiles = []
    for bursty_measure_name in bursty_measure_names:
        quantiles.append(stopwords_ranks_df[bursty_measure_name].quantile([0, 0.25, 0.5, 0.75, 1]))
    quantiles_df = pd.DataFrame(quantiles)
    all_quantiles_df.append(quantiles_df)

# Extract the column and index names from the first quantiles data frame
columns = all_quantiles_df[0].columns
index = all_quantiles_df[0].index

# Initialize empty data frames to store the mean and standard deviation values
mean_df = pd.DataFrame(index=index, columns=columns)
std_df = pd.DataFrame(index=index, columns=columns)

# Compute the mean and standard deviation of corresponding elements across all matrices
for col in columns:
    for idx in index:
        values = [matrix.at[idx, col] for matrix in all_quantiles_df]
        mean_df.at[idx, col] = np.mean(values)
        std_df.at[idx, col] = np.std(values)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [02:36<00:00,  1.57s/it]


In [35]:
# Print to console
print("Mean values:")
with pd.option_context('display.precision', 4):
    display(mean_df)
print("\nStandard deviations:")
with pd.option_context('display.precision', 4):
    display(std_df)

# Write to CSV
os.makedirs('table-11-n1', exist_ok=True)
mean_df.to_csv('table-11-n1/stopword-rank-means.csv')
std_df.to_csv('table-11-n1/stopword-rank-sds.csv')

Mean values:


,0.00,0.25,0.50,0.75,1.00
IDF,4405.2,39839.73,40595.51,40765.0,40804.0
ICF,4546.88,39749.33,40564.29,40760.0,40804.0
Chi-sq,781.32,7402.0,8077.0,8736.0,40803.0
CG,19.0,6318.0,7748.48,8619.0,39419.6
ICB,70.0,6042.0,12018.0,17067.0,36576.88
DoP,5059.01,39858.0,40602.0,40765.0,40804.0
KeyBERT,22350.0,26964.14,31420.18,36023.95,40654.89
KeyLLM,20672.19,25753.15,30787.69,35735.27,40581.58
RICF,2427.0,7804.0,8287.0,8844.0,40803.0



Standard deviations:


,0.00,0.25,0.50,0.75,1.00
IDF,3645.8235,8.8078,0.4999,0.0,0.0
ICF,3672.8082,6.53,0.4538,0.0,0.0
Chi-sq,538.7592,0.0,0.0,0.0,0.0
CG,0.0,0.0,0.4996,0.0,1132.4581
ICB,0.0,0.0,0.0,0.0,56.0122
DoP,7.7143,0.0,0.0,0.0,0.0
KeyBERT,128.6187,661.1948,749.4539,714.4045,150.2636
KeyLLM,206.4445,926.4747,880.5733,856.2755,219.6608
RICF,0.0,0.0,0.0,0.0,0.0
